# Race to Mars: Global Space Exploration Analysis
**Objective:** This notebook answers all business goals outlined in the white paper using visual analysis and predictive modeling. It includes success metrics by country, satellite and technology trends, environmental impact heatmaps, and a random forest classifier to estimate which countries are leading the global space race.

In [1]:
import sys
!{sys.executable} -m pip install seaborn plotly scikit-learn

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Load Dataset

In [3]:
df = pd.read_csv('Global_Space_Exploration_Dataset copy.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Global_Space_Exploration_Dataset copy.csv'

## Data Cleaning

In [ ]:
df['Environmental Impact'] = df['Environmental Impact'].str.strip()
df['Mission Type'] = df['Mission Type'].str.strip()
df['Country'] = df['Country'].str.strip()
df['Technology Used'] = df['Technology Used'].str.strip()

## Budget vs Success Rate by Country

In [ ]:
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='Budget (in Billion $)', y='Success Rate (%)', hue='Country', alpha=0.7)
plt.title('Budget vs. Success Rate by Country')
plt.xlabel('Budget (in Billion $)')
plt.ylabel('Success Rate (%)')
plt.grid(True)
plt.tight_layout()
plt.show()

## Mission Count by Country and Mission Type

In [ ]:
plt.figure(figsize=(14, 7))
sns.countplot(data=df, x='Country', hue='Mission Type')
plt.title('Mission Count by Country and Mission Type')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Satellite Type Distribution

In [ ]:
plt.figure(figsize=(10, 6))
df['Satellite Type'].value_counts().plot.pie(autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Satellite Types')
plt.ylabel('')
plt.tight_layout()
plt.show()

## Environmental Impact by Technology

In [ ]:
impact_tech = pd.crosstab(df['Technology Used'], df['Environmental Impact'])
plt.figure(figsize=(12, 6))
sns.heatmap(impact_tech, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Environmental Impact by Technology Type')
plt.tight_layout()
plt.show()

## Predictive Modeling: Which Country is Leading?

In [ ]:
df_model = df.copy()
label_encoders = {}
for col in ['Country', 'Mission Type', 'Technology Used', 'Satellite Type', 'Environmental Impact']:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le

top_quartile = df_model['Success Rate (%)'].quantile(0.75)
df_model['Leader'] = (df_model['Success Rate (%)'] >= top_quartile).astype(int)

features = ['Mission Type', 'Budget (in Billion $)', 'Technology Used', 'Satellite Type', 'Environmental Impact']
X = df_model[features]
y = df_model['Leader']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_prob = rf.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_prob)

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Country Leadership Prediction')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()